# Detector v5b (sin Drive)
`v5` acertaba toda la colección del profesor, pero marcaba como daño huevos sanos con sello impreso y pelotas con logotipo. `v5b` parte de `v5` y añade: sellos, fechas y logotipos sintéticos sobre huevos sanos (siguen siendo sanos), pelotas de ping-pong enteras como sanas, y menos manchas sintéticas. Mismo dataset que `v5` en lo demás. Dataset y pesos salen de la rama temporal `v5-tmp`; el resultado se descarga como zip.

In [ ]:
%pip install -q "ultralytics==8.4.161"

import json
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import requests
import yaml
import ultralytics
import torch

UA = {'User-Agent': 'eggs-detector-eval/1.0 (academic project; https://github.com/dportilla219/eggs-detector)'}


def bajar(url, destino, intentos=4):
    # Descarga una imagen (con reintentos, por si Wikimedia limita la velocidad) y comprueba que se pueda abrir.
    import time
    from PIL import Image
    if destino.exists():
        return True
    for k in range(intentos):
        try:
            r = requests.get(url, headers=UA, timeout=60)
            if r.ok:
                destino.write_bytes(r.content)
                Image.open(destino).verify()
                return True
        except Exception:
            destino.unlink(missing_ok=True)
        time.sleep(2 * (k + 1))
    print('No se pudo descargar (se omite):', url)
    return False

assert torch.cuda.is_available(), 'Este notebook necesita GPU (T4).'
TMP = Path('/content/tmp')
if not TMP.exists():
    !git clone -q --depth 1 -b v5-tmp https://github.com/dportilla219/eggs-detector {TMP}
for f in sorted(TMP.glob('*.parte1de*')):   # une los trozos de los zips
    nombre, total = f.name.rsplit('.parte1de', 1)
    with open(TMP / nombre, 'wb') as out:
        for i in range(1, int(total) + 1):
            out.write((TMP / f'{nombre}.parte{i}de{total}').read_bytes())
DRIVE_DIR = TMP
OUT_DIR = Path('/content/out')
RUNS_DIR, EXPORTS_DIR = OUT_DIR / 'runs', OUT_DIR / 'exports'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
BASE_PT = TMP / 'v2_best.pt'
INIT_PT = TMP / 'v5_best.pt'                 # v5b parte de v5
ETIQ_V3 = TMP / 'extra/etiquetas_v3fmt.json'  # mismas cajas y mismo split que v4
DATA_DIR = Path('/content/eggs_v2')          # dataset original
V2_DIR = Path('/content/eggs_v2s')           # dataset original + sintéticas (igual que en v2)
PUB_DIR = Path('/content/publico')           # fotos reales públicas en formato YOLO
BASE_RUN, PREV_RUN, NEW_RUN = 'v2', 'v5', 'v5b'
CLASSES = ['Crack', 'Intact']
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not (DATA_DIR / 'train').exists():
    for n in ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip',
              'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']:
        with zipfile.ZipFile(DRIVE_DIR / n) as zf:
            zf.extractall(DATA_DIR)
        print('OK', n)
with open(DATA_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump({'path': str(DATA_DIR), 'train': str(DATA_DIR / 'train/images'),
                    'val': str(DATA_DIR / 'valid/images'), 'test': str(DATA_DIR / 'test/images'),
                    'nc': 2, 'names': CLASSES}, f, sort_keys=False)

libre, total = torch.cuda.mem_get_info()
print(f'GPU: {torch.cuda.get_device_name(0)}, libre {libre / 1e9:.1f} de {total / 1e9:.1f} GB')
print('Pesos de v2:', BASE_PT)
print('Resultados en:', OUT_DIR)

In [ ]:
import random
import shutil
from collections import Counter

import cv2
import numpy as np


def fuente(stem):
    """'montaje' = fotos del montaje de fondo gris (todas las Intact y parte de las Crack)."""
    return 'montaje' if stem.startswith('ec_egg') else 'otras'


def cargar_split(split_dir):
    """Lista de dicts {img, cls, box (x1,y1,x2,y2 en px), fuente} de las imágenes originales (sin _dup)."""
    items = []
    for img in sorted((split_dir / 'images').iterdir()):
        if img.suffix.lower() not in IMG_EXTS or '_dup' in img.stem:
            continue
        lines = (split_dir / 'labels' / f'{img.stem}.txt').read_text().split('\n')
        parts = lines[0].split()
        if len(parts) != 5:
            continue
        h, w = cv2.imread(str(img)).shape[:2]
        c, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        items.append({'img': img, 'cls': c, 'box': box, 'fuente': fuente(img.stem), 'size': (w, h)})
    return items


def recortar_huevo(item):
    """Recorta el huevo con una máscara elíptica suavizada (los huevos son casi elipses)."""
    img = cv2.imread(str(item['img']))
    x1, y1, x2, y2 = [int(round(v)) for v in item['box']]
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, img.shape[1]), min(y2, img.shape[0])
    crop = img[y1:y2, x1:x2].copy()
    h, w = crop.shape[:2]
    mask = np.zeros((h, w), np.float32)
    cv2.ellipse(mask, (w // 2, h // 2), (max(w // 2 - 1, 1), max(h // 2 - 1, 1)), 0, 0, 360, 1.0, -1)
    k = max(3, (min(w, h) // 25) | 1)
    mask = cv2.GaussianBlur(mask, (k, k), 0)
    return crop, mask


def variar_huevo(crop, rng):
    """Volteo aleatorio y, a veces, 'blanquear' el huevo (hay huevos blancos sanos en la vida real)."""
    if rng.random() < 0.5:
        crop = crop[:, ::-1]
    if rng.random() < 0.3:
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(0.1, 0.4)
        hsv[..., 2] = np.clip(hsv[..., 2] * rng.uniform(1.1, 1.4) + 20, 0, 255)
        crop = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return np.ascontiguousarray(crop)


def pegar(bg, crop, mask, cx, cy, w, h):
    """Pega el huevo (w x h px) centrado en (cx, cy). Devuelve la caja visible o None si queda muy cortado."""
    w, h = max(int(w), 4), max(int(h), 4)
    crop = cv2.resize(crop, (w, h), interpolation=cv2.INTER_AREA if w < crop.shape[1] else cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (w, h))[..., None]
    H, W = bg.shape[:2]
    x1, y1 = int(cx - w / 2), int(cy - h / 2)
    bx1, by1, bx2, by2 = max(x1, 0), max(y1, 0), min(x1 + w, W), min(y1 + h, H)
    if (bx2 - bx1) * (by2 - by1) < 0.85 * w * h:
        return None
    roi = bg[by1:by2, bx1:bx2].astype(np.float32)
    c = crop[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1].astype(np.float32)
    m = mask[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1]
    bg[by1:by2, bx1:bx2] = (c * m + roi * (1 - m)).astype(np.uint8)
    return bx1, by1, bx2, by2


def fondo_procedural(rng, W, H):
    """Fondos variados (color liso con ruido, degradado o textura) que no pertenecen a ninguna clase."""
    tipo = rng.randrange(3)
    c1 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
    if tipo == 0:
        bg = np.ones((H, W, 3), np.float32) * c1
    elif tipo == 1:
        c2 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
        t = np.linspace(0, 1, W if rng.random() < 0.5 else H, dtype=np.float32)
        t = t[None, :, None] if len(t) == W else t[:, None, None]
        bg = np.broadcast_to(c1 * (1 - t) + c2 * t, (H, W, 3)).copy()
    else:
        ruido = np.random.default_rng(rng.randrange(1 << 30)).random((H // 16 + 1, W // 16 + 1, 1)).astype(np.float32)
        ruido = cv2.resize(ruido, (W, H), interpolation=cv2.INTER_CUBIC)[..., None]
        bg = c1 * (0.6 + 0.8 * ruido)
    bg += np.random.default_rng(rng.randrange(1 << 30)).normal(0, rng.uniform(2, 12), (H, W, 3)).astype(np.float32)
    return np.clip(bg, 0, 255).astype(np.uint8)


def mira_verde(img, rng):
    """La mira verde del montaje, dibujada en imágenes de ambas clases para que no sea pista de Intact."""
    H, W = img.shape[:2]
    cx, cy = int(W * rng.uniform(0.4, 0.6)), int(H * rng.uniform(0.4, 0.6))
    color = (int(rng.uniform(80, 140)), int(rng.uniform(170, 230)), int(rng.uniform(80, 140)))
    cv2.line(img, (0, cy), (W, cy), color, 1)
    cv2.line(img, (cx, 0), (cx, H), color, 1)


def retocar(img, cajas, rng):
    """Mira verde en ambas clases y, a veces, bajar la resolución de toda la imagen para que la nitidez
    tampoco delate la clase (los huevos Intact vienen de fotos de 224 px)."""
    if rng.random() < 0.4:
        mira_verde(img, rng)
    if rng.random() < 0.5:
        H, W = img.shape[:2]
        f = rng.uniform(224, 320) / max(H, W)
        if f < 1:
            img = cv2.resize(img, (max(int(W * f), 1), max(int(H * f), 1)), interpolation=cv2.INTER_AREA)
            cajas = [(c, x1 * f, y1 * f, x2 * f, y2 * f) for c, x1, y1, x2, y2 in cajas]
    return img, cajas


def guardar(out_dir, nombre, img, cajas):
    """cajas: lista de (cls, x1, y1, x2, y2) en px -> .jpg + .txt YOLO."""
    H, W = img.shape[:2]
    cv2.imwrite(str(out_dir / 'images' / f'{nombre}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    lineas = [f'{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
              for c, x1, y1, x2, y2 in cajas]
    (out_dir / 'labels' / f'{nombre}.txt').write_text('\n'.join(lineas) + '\n')


def intercambio(fondo_item, huevo_item, rng):
    """Tapa el huevo del fondo con el otro huevo (un 12-25 % más grande para cubrirlo entero)."""
    bg = cv2.imread(str(fondo_item['img']))
    crop, mask = recortar_huevo(huevo_item)
    crop = variar_huevo(crop, rng)
    x1, y1, x2, y2 = fondo_item['box']
    ch, cw = crop.shape[:2]
    s = max((x2 - x1) / cw, (y2 - y1) / ch) * rng.uniform(1.12, 1.25)
    return bg, pegar(bg, crop, mask, (x1 + x2) / 2, (y1 + y2) / 2, cw * s, ch * s)


def procedural(huevos, rng):
    """1 o 2 huevos sobre un fondo procedural, con tamaño y posición al azar."""
    W, H = rng.choice([(640, 640), (640, 480), (480, 640), (288, 640)])
    bg = fondo_procedural(rng, W, H)
    cajas = []
    for _ in range(2 if rng.random() < 0.3 else 1):
        item = rng.choice(huevos)
        crop, mask = recortar_huevo(item)
        crop = variar_huevo(crop, rng)
        ch, cw = crop.shape[:2]
        s = rng.uniform(0.25, 0.7) * min(W, H) / max(ch, cw)
        w, h = cw * s, ch * s
        for _intento in range(10):
            cx, cy = rng.uniform(w / 2, W - w / 2), rng.uniform(h / 2, H - h / 2)
            if all(cx + w / 2 < a or cx - w / 2 > c or cy + h / 2 < b or cy - h / 2 > d for _, a, b, c, d in cajas):
                caja = pegar(bg, crop, mask, cx, cy, w, h)
                if caja:
                    cajas.append((item['cls'], *caja))
                break
    return bg, cajas


def generar_sinteticas(split_dir, out_dir, n_a, n_b, n_p, seed, prefijo):
    """A: Intact del montaje sobre fondos de otras fuentes. B: Crack de otras fuentes sobre el montaje.
    P: huevos de ambas clases sobre fondos procedurales. Solo usa huevos y fondos del mismo split."""
    rng = random.Random(seed)
    items = cargar_split(split_dir)
    intact = [i for i in items if i['cls'] == 1]
    crack_otras = [i for i in items if i['cls'] == 0 and i['fuente'] == 'otras']
    montaje = [i for i in items if i['fuente'] == 'montaje']
    (out_dir / 'images').mkdir(parents=True, exist_ok=True)
    (out_dir / 'labels').mkdir(parents=True, exist_ok=True)
    conteo = Counter()
    tareas = [('A', n_a, crack_otras, intact), ('B', n_b, montaje, crack_otras)]
    for tipo, n, fondos, huevos in tareas:
        hechos = 0
        orden = (huevos * (n // len(huevos) + 1))[:n] if tipo == 'A' else [rng.choice(huevos) for _ in range(n)]
        for k, huevo in enumerate(orden):
            for _intento in range(5):
                bg, caja = intercambio(rng.choice(fondos), huevo, rng)
                if caja:
                    break
            if not caja:
                continue
            bg, cajas = retocar(bg, [(huevo['cls'], *caja)], rng)
            guardar(out_dir, f'{prefijo}_{tipo}_{k:05d}', bg, cajas)
            conteo[(tipo, CLASSES[huevo['cls']])] += 1
            hechos += 1
    # mitad Intact / mitad Crack para que el fondo procedural no favorezca a ninguna clase
    pools = [intact, [i for i in items if i['cls'] == 0]]
    for k in range(n_p):
        bg, cajas = procedural(pools[k % 2], rng)
        if not cajas:
            continue
        bg, cajas = retocar(bg, cajas, rng)
        guardar(out_dir, f'{prefijo}_P_{k:05d}', bg, cajas)
        for c, *_ in cajas:
            conteo[('P', CLASSES[c])] += 1
    return conteo

In [ ]:
import pandas as pd

if V2_DIR.exists():
    shutil.rmtree(V2_DIR)

def copiar_originales(split, destino):
    for sub in ['images', 'labels']:
        (destino / sub).mkdir(parents=True, exist_ok=True)
    for img in (DATA_DIR / split / 'images').iterdir():
        if img.suffix.lower() in IMG_EXTS and '_dup' not in img.stem:
            shutil.copy2(img, destino / 'images' / img.name)
            shutil.copy2(DATA_DIR / split / 'labels' / f'{img.stem}.txt', destino / 'labels' / f'{img.stem}.txt')

conteos = {}
copiar_originales('train', V2_DIR / 'train')
conteos['train'] = generar_sinteticas(DATA_DIR / 'train', V2_DIR / 'train', 1228, 700, 700, seed=0, prefijo='syn_train')
copiar_originales('valid', V2_DIR / 'valid')
conteos['valid'] = generar_sinteticas(DATA_DIR / 'valid', V2_DIR / 'valid', 356, 200, 200, seed=1, prefijo='syn_valid')
conteos['test_synth'] = generar_sinteticas(DATA_DIR / 'test', V2_DIR / 'test_synth', 168, 168, 200, seed=2, prefijo='syn_test')

def escribir_yaml(nombre, **splits):
    p = V2_DIR / nombre
    with open(p, 'w') as f:
        yaml.safe_dump({'path': str(V2_DIR), **splits, 'nc': 2, 'names': CLASSES}, f, sort_keys=False)
    return p

DATA_V2 = escribir_yaml('data.yaml', train=str(V2_DIR / 'train/images'), val=str(V2_DIR / 'valid/images'),
                        test=str(DATA_DIR / 'test/images'))
DATA_TEST_ALL = escribir_yaml('test_all.yaml', train=str(V2_DIR / 'train/images'),
                              val=[str(DATA_DIR / 'test/images'), str(V2_DIR / 'test_synth/images')])

def resumen(split_dir):
    c = {}
    for lp in (split_dir / 'labels').glob('*.txt'):
        for line in lp.read_text().splitlines():
            if line.strip():
                k = CLASSES[int(line.split()[0])]
                c[k] = c.get(k, 0) + 1
    return c

filas = {s: resumen(V2_DIR / s) for s in ['train', 'valid', 'test_synth']}
display(pd.DataFrame(filas).T.assign(ratio=lambda d: (d['Crack'] / d['Intact']).round(2)))
print({s: dict(c) for s, c in conteos.items()})

In [ ]:
from ultralytics import YOLO

PUB_SRC = Path('/content/publico_src')
if not PUB_SRC.exists():
    !git clone -q --depth 1 https://github.com/dakshkathuria346-gif/Egg-Defect-Detection {PUB_SRC}

COMMONS_TRAIN = [
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b7/2_Huevos_de_distintas_especies_de_gallinas_21.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas_21.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas 21.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  292
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/86/2_Huevos_de_distintas_especies_de_gallinas_7.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas_7.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas 7.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  293
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ec/2_Huevos_de_distintas_especies_de_gallinas.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  294
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5b/20170602-AMS-PJK-0101_%2834890669702%29.jpg/960px-20170602-AMS-PJK-0101_%2834890669702%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:20170602-AMS-PJK-0101 (34890669702).jpg",
  "Public domain",
  "U.S. Department of Agriculture\n\nPreston Keres/Office of Communications-Photograp",
  298
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/23-365_-_Flickr_-_eren_%28sea-prairie%29.jpg/960px-23-365_-_Flickr_-_eren_%28sea-prairie%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:23-365 - Flickr - eren (sea-prairie).jpg",
  "CC BY-SA 2.0",
  "eren {sea+prairie}",
  301
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1f/A_Clutch_of_White_Bird_Eggs_Nestled_in_Dried_Grass.jpg/960px-A_Clutch_of_White_Bird_Eggs_Nestled_in_Dried_Grass.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:A Clutch of White Bird Eggs Nestled in Dried Grass.jpg",
  "CC BY-SA 4.0",
  "A S M Jobaer",
  312
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/69/An_Egg_%2832970626574%29.jpg/960px-An_Egg_%2832970626574%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:An Egg (32970626574).jpg",
  "CC BY 2.0",
  "NIAID",
  330
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5e/An_Egg_%2833684416751%29.jpg/960px-An_Egg_%2833684416751%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:An Egg (33684416751).jpg",
  "CC BY 2.0",
  "NIAID",
  331
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/9/95/Anne_Biermann_eier.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Anne Biermann eier.jpg",
  "Public domain",
  "aenne biermann",
  335
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/45/At_London_2025_112.jpg/960px-At_London_2025_112.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:At London 2025 112.jpg",
  "CC BY-SA 4.0",
  "Photograph by Mike Peel (www.mikepeel.net).",
  337
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/ad/At_London_2025_113.jpg/960px-At_London_2025_113.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:At London 2025 113.jpg",
  "CC BY-SA 4.0",
  "Photograph by Mike Peel (www.mikepeel.net).",
  338
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6d/Barred_Plymouth_Rock_egg_%28cropped%29.jpg/960px-Barred_Plymouth_Rock_egg_%28cropped%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Barred Plymouth Rock egg (cropped).jpg",
  "CC BY-SA 3.0",
  "",
  343
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6f/Barred_Plymouth_Rock_egg.jpg/960px-Barred_Plymouth_Rock_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Barred Plymouth Rock egg.jpg",
  "CC BY-SA 3.0",
  "",
  344
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/52/Blue_hen_eggs.jpg/960px-Blue_hen_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Blue hen eggs.jpg",
  "CC BY-SA 4.0",
  "Secretlondon",
  346
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Boiled-Egg-spoons-with-egg.jpg/960px-Boiled-Egg-spoons-with-egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Boiled-Egg-spoons-with-egg.jpg",
  "CC BY-SA 4.0",
  "Grenadille",
  350
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Braunes_H%C3%BChnerei.JPG/960px-Braunes_H%C3%BChnerei.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Braunes Hühnerei.JPG",
  "CC BY-SA 3.0",
  "Fiver, der Hellseher",
  351
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0f/Brown_Egg_%2818423463472%29.jpg/960px-Brown_Egg_%2818423463472%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Brown Egg (18423463472).jpg",
  "CC BY-SA 2.0",
  "Willis Lam",
  352
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Chicken%27s_egg.jpg/960px-Chicken%27s_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken's egg.jpg",
  "CC BY-SA 4.0",
  "Chris Prince Udochukwu Njoku",
  374
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/79/Comparativa_huevo_de_gallina_y_de_avestruz.jpg/960px-Comparativa_huevo_de_gallina_y_de_avestruz.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Comparativa huevo de gallina y de avestruz.jpg",
  "CC BY-SA 4.0",
  "Kbemcap",
  379
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c8/Country_chicken_eggs.jpg/960px-Country_chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Country chicken eggs.jpg",
  "CC BY-SA 4.0",
  "சு.பத்மா",
  380
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Country_Eggs_in_Salem.jpg/960px-Country_Eggs_in_Salem.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Country Eggs in Salem.jpg",
  "CC BY-SA 4.0",
  "Thamizhpparithi Maari",
  381
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Egg_High_Key_%285452144772%29.jpg/960px-Egg_High_Key_%285452144772%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg High Key (5452144772).jpg",
  "CC BY 2.0",
  "Nolan Williamson from Atlanta, GA, United States of America",
  394
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a3/Egg_KyakOo.jpg/960px-Egg_KyakOo.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg KyakOo.jpg",
  "CC BY-SA 4.0",
  "SarKaLay စာကလေး",
  397
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/4/44/Egg_stand.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Egg stand.jpg",
  "CC BY-SA 2.0",
  "☈ASCAL",
  400
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/53/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_01.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 01.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  413
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/36/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_04.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_04.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 04.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  416
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/45/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_05.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_05.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 05.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  417
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/9/9a/Ei01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Ei01.jpg",
  "Public domain",
  "No machine-readable author provided. Acf assumed (based on copyright claims).",
  424
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Ei_und_H%C3%BChnerfeder_2.jpg/960px-Ei_und_H%C3%BChnerfeder_2.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 2.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  425
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7b/Ei_und_H%C3%BChnerfeder_5a.jpg/960px-Ei_und_H%C3%BChnerfeder_5a.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 5a.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  428
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d8/Ei_und_H%C3%BChnerfeder_5b.jpg/960px-Ei_und_H%C3%BChnerfeder_5b.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 5b.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  429
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4a/Eier_in_Eierkarton_5.jpg/960px-Eier_in_Eierkarton_5.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier in Eierkarton 5.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  436
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8e/Eier_in_Eierkarton_6.jpg/960px-Eier_in_Eierkarton_6.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier in Eierkarton 6.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  437
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/cf/Gallus_gallus_MWNH_1135.JPG/960px-Gallus_gallus_MWNH_1135.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Gallus gallus MWNH 1135.JPG",
  "CC BY-SA 3.0",
  "Klaus Rassinger und Gerhard Cammerer, Museum Wiesbaden",
  493
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c7/Hand_holding_a_fresh_egg_from_a_chicken_coop.jpg/960px-Hand_holding_a_fresh_egg_from_a_chicken_coop.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Hand holding a fresh egg from a chicken coop.jpg",
  "CC BY 2.0",
  "Shixart1985",
  496
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2f/Hand_holds_a_raw_white_egg_in_a_dimly_lit_kitchen.jpg/960px-Hand_holds_a_raw_white_egg_in_a_dimly_lit_kitchen.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Hand holds a raw white egg in a dimly lit kitchen.jpg",
  "CC BY 2.0",
  "Shixart1985",
  497
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9e/Jajci_s_slovenskimi_oznakami.jpg/960px-Jajci_s_slovenskimi_oznakami.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Jajci s slovenskimi oznakami.jpg",
  "CC BY-SA 4.0",
  "Melaleuca alternifolia",
  510
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/8/8a/Koni-juj.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Koni-juj.jpg",
  "CC BY-SA 4.0",
  "Parikhit phukan",
  514
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/aa/Kwai_na_kaza.jpg/960px-Kwai_na_kaza.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Kwai na kaza.jpg",
  "CC0",
  "Salaha mahmood",
  518
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6c/Kwan_kaza.jpg/960px-Kwan_kaza.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Kwan kaza.jpg",
  "CC0",
  "Umar A Muhammad",
  519
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4e/Meat_serves.jpg/960px-Meat_serves.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Meat serves.jpg",
  "CC BY-SA 3.0",
  "A8younan",
  529
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Multi-Colored_Eggs_%288065133518%29.jpg/960px-Multi-Colored_Eggs_%288065133518%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Multi-Colored Eggs (8065133518).jpg",
  "CC BY 2.0",
  "Larry Lamsa",
  533
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ea/Ous_falsos.jpg/960px-Ous_falsos.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ous falsos.jpg",
  "CC BY-SA 4.0",
  "GallRC",
  539
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/f/fd/Pigmentation_of_eggs._Incomplete_pigmentation_in_the_color_of_a_chicken_egg_shell._81.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Pigmentation of eggs. Incomplete pigmentation in the color of a chicken egg shell. 81.jpg",
  "CC BY-SA 4.0",
  "ВоєводаЯ",
  546
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/1/17/Pigmentation_of_eggs._Incomplete_pigmentation_in_the_color_of_a_chicken_egg_shell._83.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Pigmentation of eggs. Incomplete pigmentation in the color of a chicken egg shell. 83.jpg",
  "CC BY-SA 4.0",
  "ВоєводаЯ",
  548
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7c/Ramen_%40_Ichiran_%40_Shinjuku_%2815248167521%29.jpg/960px-Ramen_%40_Ichiran_%40_Shinjuku_%2815248167521%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ramen @ Ichiran @ Shinjuku (15248167521).jpg",
  "CC BY 2.0",
  "Guilhem Vellut from Annecy, France",
  556
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/eb/Silverudds_bl%C3%A5_%C3%A4gg.jpg/960px-Silverudds_bl%C3%A5_%C3%A4gg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Silverudds blå ägg.jpg",
  "CC BY-SA 4.0",
  "Ceciliaw79",
  562
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f1/Three_eggs_captured_in_May_2022.jpg/960px-Three_eggs_captured_in_May_2022.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Three eggs captured in May 2022.jpg",
  "CC BY-SA 4.0",
  "Billjones94",
  569
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0e/Two_cups.jpg/960px-Two_cups.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Two cups.jpg",
  "CC BY 2.0",
  "darwin Bell from San Francisco, USA",
  574
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b0/%E1%80%80%E1%80%BC%E1%80%80%E1%80%BA%E1%80%A5.jpg/960px-%E1%80%80%E1%80%BC%E1%80%80%E1%80%BA%E1%80%A5.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:ကြက်ဥ.jpg",
  "CC BY-SA 4.0",
  "SarKaLay စာကလေး",
  595
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3f/%E9%B8%A1%E8%9B%8B%EF%BC%88%E7%A9%BA%EF%BC%89_-_panoramio.jpg/960px-%E9%B8%A1%E8%9B%8B%EF%BC%88%E7%A9%BA%EF%BC%89_-_panoramio.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:鸡蛋（空） - panoramio.jpg",
  "CC BY-SA 3.0",
  "zhanyoun",
  600
 ]
]
CT_DIR = Path('/content/commons_train')
CT_DIR.mkdir(exist_ok=True)
for i, (url, *_resto) in enumerate(COMMONS_TRAIN):
    bajar(url, CT_DIR / f'{i:03d}.jpg')

from PIL import Image as PILImage, ImageOps


def orden_natural(nombre):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', nombre)]


def subgrupo(nombre):
    # 'Egg_X12_jpeg.rf...' -> 'Egg_X', 'damaged_613_jpg...' -> 'damaged', '000.jpg' (Commons) -> 'commons'
    return 'commons' if nombre[:1].isdigit() else re.sub(r'[_-]?\d.*$', '', nombre)


etiquetas = []
for e in json.load(open(ETIQ_V3)):
    origen = 'wtr' if e['archivo'][:1].isdigit() else 'pub'
    clase = CLASSES.index(e['clase'])
    f = CT_DIR / e['archivo'] if origen == 'wtr' else PUB_SRC / ('non defective' if clase == 1 else 'defective') / e['archivo']
    if not e['aceptada'] or not f.exists():
        continue
    W, H = ImageOps.exif_transpose(PILImage.open(f)).size
    etiquetas.append({'file': f, 'clase': clase, 'W': W, 'H': H, 'cajas': e['cajas'], 'ok': True, 'origen': origen,
                      'split_v3': e['split'], 'grupo': (origen, clase, subgrupo(e['archivo']))})

for g in sorted({e['grupo'] for e in etiquetas}, key=str):
    filas = sorted([e for e in etiquetas if e['grupo'] == g], key=lambda e: orden_natural(e['file'].name))
    libres = [e for e in filas if e['split_v3'] != 'train']   # fotos que v3 no usó para entrenar
    for e in filas:
        e['split'] = 'train' if e['split_v3'] == 'train' else None
    for i, e in enumerate(libres):
        e['split'] = ('train', 'train', 'valid', 'test')[(i // 4) % 4]

# Fotos extra de v5b: pelotas de ping-pong enteras con logotipo (sanas) y una rajada, de Wikimedia Commons
for x in json.load(open(TMP / 'extra/extra.json', encoding='utf-8')):
    etiquetas.append({'file': TMP / 'extra' / x['archivo'], 'clase': x['clase'], 'W': x['W'], 'H': x['H'], 'cajas': x['cajas'], 'ok': True,
                      'origen': 'ext', 'split_v3': 'train', 'split': 'train', 'grupo': ('ext', x['clase'], 'pelotas')})

if PUB_DIR.exists():
    shutil.rmtree(PUB_DIR)


def guardar_yolo(e, split, nombre):
    for sub in ('images', 'labels'):
        (PUB_DIR / split / sub).mkdir(parents=True, exist_ok=True)
    shutil.copy2(e['file'], PUB_DIR / split / 'images' / f'{nombre}{e["file"].suffix.lower()}')
    W, H = e['W'], e['H']
    (PUB_DIR / split / 'labels' / f'{nombre}.txt').write_text('\n'.join(
        f'{e["clase"]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
        for x1, y1, x2, y2 in e['cajas']) + '\n')


conteo = Counter()
for e in etiquetas:
    copias = 3 if e['split'] == 'train' else 1  # las fotos reales pesan más
    for k in range(copias):
        guardar_yolo(e, e['split'], f'{e["origen"]}_{CLASSES[e["clase"]]}_{e["file"].stem}' + (f'_r{k}' if k else ''))
    conteo[(e['grupo'][2], CLASSES[e['clase']], e['split'])] += 1
pub_ok = Counter(f"{e['origen']} {CLASSES[e['clase']]}" for e in etiquetas)
pub_total = pub_ok
print('Fotos reales por subgrupo y split (sin repeticiones):')
for k in sorted(conteo, key=str):
    print('  ', k, conteo[k])

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

AUD_DIR = RUNS_DIR / 'v5_auditoria'
AUD_DIR.mkdir(parents=True, exist_ok=True)


def hoja(filas, destino, T=200, cols=8):
    rows = max(1, (len(filas) + cols - 1) // cols)
    lienzo = np.full((rows * T, cols * T, 3), 30, np.uint8)
    for i, e in enumerate(filas):
        img = cv2.imread(str(e['file']))
        k = T / max(img.shape[:2])
        img = cv2.resize(img, (max(1, int(img.shape[1] * k)), max(1, int(img.shape[0] * k))))
        color = (0, 200, 0) if e['clase'] == 1 else (0, 0, 255)
        for x1, y1, x2, y2 in e['cajas']:
            cv2.rectangle(img, (int(x1 * k), int(y1 * k)), (int(x2 * k), int(y2 * k)), color, 2)
        cv2.putText(img, e['split'][:2], (3, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        y, x = (i // cols) * T, (i % cols) * T
        lienzo[y:y + img.shape[0], x:x + img.shape[1]] = img
    cv2.imwrite(str(destino), lienzo, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return lienzo


nuevas = [e for e in etiquetas if e['split'] == 'train' and e['split_v3'] != 'train']
print(f'{len(nuevas)} fotos reales nuevas en train (antes en valid/test de v3)')
ejemplo = hoja(nuevas[::max(1, len(nuevas) // 32)][:32], AUD_DIR / 'nuevas_en_train.jpg')
plt.figure(figsize=(18, 9)); plt.imshow(ejemplo[..., ::-1]); plt.axis('off'); plt.show()
json.dump([{'archivo': e['file'].name, 'clase': CLASSES[e['clase']], 'split': e['split'], 'split_v3': e['split_v3'],
            'grupo': e['grupo'][2], 'cajas': e['cajas']} for e in etiquetas], open(AUD_DIR / 'etiquetas.json', 'w'), indent=0)
print('Guardado en', AUD_DIR)

In [ ]:
%pip install -q ai-edge-litert

REPO = Path('/content/eggs-detector')
if not REPO.exists():
    !git clone -q --depth 1 https://github.com/dportilla219/eggs-detector {REPO}
PROFE_SRC = Path('/content/huevos_profe')
if not PROFE_SRC.exists():
    !git clone -q https://github.com/adiacla/huevos {PROFE_SRC}
    !git -C {PROFE_SRC} checkout -q 1b4e76826a6c1abfe0a4f20bb16f08b108569012

PROFE_DIR = Path('/content/profe')
json.dump([{'file': str(e['file']), 'clase': e['clase'], 'cajas': e['cajas'], 'split': e['split']} for e in etiquetas],
          open('/content/reales.json', 'w'))
if PROFE_DIR.exists():
    shutil.rmtree(PROFE_DIR)
!python {REPO}/v5/datos.py --profe {PROFE_SRC}/imagenes --reales /content/reales.json --dano {REPO}/modelo/eggs_dano_v1_fp16.tflite --out {PROFE_DIR} --n 2600 300 300
for s in ['train', 'valid', 'test', 'originales']:
    print(s, len(list((PROFE_DIR / s / 'images').iterdir())), 'imágenes')

# Los rajados del montaje original (ec_egg*), una copia más en train
MONT_DIR = Path('/content/montaje_extra')
if MONT_DIR.exists():
    shutil.rmtree(MONT_DIR)
for sub in ('images', 'labels'):
    (MONT_DIR / sub).mkdir(parents=True)
n_mont = 0
for img in (DATA_DIR / 'train/images').iterdir():
    lab = DATA_DIR / 'train/labels' / f'{img.stem}.txt'
    if img.stem.startswith('ec_egg') and '_dup' not in img.stem and lab.exists() and any(l.split()[:1] == ['0'] for l in lab.read_text().splitlines()):
        shutil.copy2(img, MONT_DIR / 'images' / f'{img.stem}_m1{img.suffix}')
        shutil.copy2(lab, MONT_DIR / 'labels' / f'{img.stem}_m1.txt')
        n_mont += 1
print('rajados del montaje repetidos:', n_mont)

# muestra de composiciones con sus cajas (rojo = con daño, verde = sano)
muestra = sorted((PROFE_DIR / 'train/images').glob('prf_train_*.jpg'))[:12]
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for ax, f in zip(axes.flat, muestra):
    img = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for l in (PROFE_DIR / 'train/labels' / f'{f.stem}.txt').read_text().splitlines():
        c, xc, yc, w, h = map(float, l.split())
        cv2.rectangle(img, (int((xc - w / 2) * W), int((yc - h / 2) * H)), (int((xc + w / 2) * W), int((yc + h / 2) * H)),
                      (220, 0, 0) if c == 0 else (0, 170, 0), max(2, W // 250))
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
V3_YAML = V2_DIR / 'data_v5b.yaml'
with open(V3_YAML, 'w') as f:
    yaml.safe_dump({'path': str(V2_DIR),
                    'train': [str(V2_DIR / 'train/images'), str(PUB_DIR / 'train/images'), str(PROFE_DIR / 'train/images'), str(MONT_DIR / 'images')],
                    'val': [str(V2_DIR / 'valid/images'), str(PUB_DIR / 'valid/images'), str(PROFE_DIR / 'valid/images')],
                    'test': str(DATA_DIR / 'test/images'), 'nc': 2, 'names': CLASSES}, f, sort_keys=False)

run_dir = RUNS_DIR / NEW_RUN
if (run_dir / 'results.png').exists():  # ultralytics solo crea results.png al terminar
    print(f'{NEW_RUN} ya está entrenado en {run_dir}: no se vuelve a entrenar.')
elif (run_dir / 'weights/last.pt').exists():
    print(f'{NEW_RUN} quedó a medias: se reanuda.')
    YOLO(str(run_dir / 'weights/last.pt')).train(resume=True)
else:
    assert not run_dir.exists(), f'{run_dir} existe pero no tiene pesos: revísalo (no se sobrescribe ningún run).'
    YOLO(str(INIT_PT)).train(
        data=str(V3_YAML),
        imgsz=640, epochs=14, patience=8, batch=-1, cache='disk',
        project=str(RUNS_DIR), name=NEW_RUN,
        hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.5,
        fliplr=0.5, flipud=0.2,
        mosaic=1.0, mixup=0.1,
    )

In [ ]:
import torch
import ultralytics.utils.export.litert as litert_mod
from ultralytics.utils.export.engine import _NormalizeCoords

_torch2litert_original = litert_mod.torch2litert

class _EntradaNHWC(torch.nn.Module):
    """Recibe [1, H, W, 3] (lo que entrega la cámara) y lo pasa al modelo como [1, 3, H, W]."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x.permute(0, 3, 1, 2))

class _CalibracionNHWC:
    """Pasa las imágenes de calibración INT8 a NHWC para que coincidan con la nueva entrada."""
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            batch = dict(batch)
            batch['img'] = batch['img'].permute(0, 2, 3, 1)
            yield batch

def _torch2litert_nhwc(model, im, file, quantize, calibration_dataset, metadata, prefix):
    h, w = int(im.shape[2]), int(im.shape[3])
    model = _EntradaNHWC(_NormalizeCoords(model, h, w, 'detect', len((metadata or {}).get('names', {})), None))
    normalize_original = litert_mod._NormalizeCoords
    litert_mod._NormalizeCoords = lambda m, *a, **k: m  # ya normalizado arriba
    try:
        return _torch2litert_original(
            model, im.permute(0, 2, 3, 1).contiguous(), file, quantize,
            _CalibracionNHWC(calibration_dataset) if calibration_dataset is not None else None, metadata, prefix)
    finally:
        litert_mod._NormalizeCoords = normalize_original

litert_mod.torch2litert = _torch2litert_nhwc


import gc
gc.collect(); torch.cuda.empty_cache()
out = Path('/content/eggs_v5b_resultados'); shutil.rmtree(out, ignore_errors=True); out.mkdir()
for nombre, pt in [('v5b', 'best.pt'), ('v5b_ultima', 'last.pt')]:
    w = Path('/content/exp') / nombre; shutil.rmtree(w, ignore_errors=True); w.mkdir(parents=True)
    shutil.copy2(RUNS_DIR / NEW_RUN / 'weights' / pt, w / f'eggs_{nombre}.pt')
    f = Path(YOLO(str(w / f'eggs_{nombre}.pt')).export(format='litert', imgsz=640))
    shutil.copy2(f, out / f'eggs_{nombre}_fp32.tflite'); shutil.copy2(w / f'eggs_{nombre}.pt', out / f'eggs_{nombre}.pt')
for f in ['results.csv', 'results.png', 'args.yaml']:
    if (RUNS_DIR / NEW_RUN / f).exists():
        shutil.copy2(RUNS_DIR / NEW_RUN / f, out / f)
z = shutil.make_archive(str(out), 'zip', out)
from google.colab import files
files.download(z)